# Applied Programming for AI — Mini-project 1
## Service Ticket Data Pipeline

**Student:** Mohamed Jlidi | **Student number:** a22601339

This notebook follows all **8 tasks** of the lecturer's brief. Code is deliberately split into small, reusable functions to make it easy to explain during the 5-minute oral defence.

**Goal:** keep trustworthy tickets, explain rejected rows, fix repairable values, create features, and export an audit trail. No machine-learning model is required.

## Import libraries and load the ORIGINAL CSV



In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# Try the current folder and the lecturer's folder structure.
choices = [Path("service_tickets.csv"), Path("prj/Project-1/service_tickets.csv")]
source = next((p for p in choices if p.is_file()), None)

if source is None:
    try:
        from google.colab import files
        print("Please upload service_tickets.csv (NOT titanic.csv).")
        files.upload()
        source = Path("service_tickets.csv")
    except ImportError:
        raise FileNotFoundError("Place service_tickets.csv next to this notebook.")

raw = pd.read_csv(source)
raw_before = raw.copy(deep=True)  # used to verify we do not modify it
print("Loaded:", source)
display(raw.head())

Please upload service_tickets.csv (NOT titanic.csv).


Saving service_tickets.csv to service_tickets (1).csv
Loaded: service_tickets.csv


,ticket_id,opened_on,channel,priority,handling_minutes,satisfaction
0,T001,2026-10-01,email,medium,42.2,2.0
1,T002,2026-10-01,phone,high,76.3,3.0
2,T003,2026-10-01,web,low,NaN,4.0
3,T004,2026-10-01,email,medium,118.0,5.0
4,T005,2026-10-02,phone,high,99.5,1.0


## 1. Diagnosis

**Observation unit:** one row represents **one service ticket**, identified by `ticket_id`. A repeated ID is suspicious because it can refer to two different records.

In [12]:

# 1. Display the number of rows and columns
print("Shape (rows, columns):", raw.shape)

# 2. Show the data type of each column
print("\nColumn types:")
print(raw.dtypes)

# 3. Count missing values in each column
print("\nMissing values per column:")
print(raw.isna().sum())

# 4. Count rows that are exactly identical
print("\nExact duplicate rows:", raw.duplicated().sum())

# 5. Count rows that share the same ticket ID
print("Repeated ticket IDs (rows involved):",
      raw.duplicated(subset=["ticket_id"], keep=False).sum()) #Mark ALL occurrences of a repeated ticket ID as duplicates, including the first one.

# 6. Display all rows with repeated ticket IDs
display(raw[raw.duplicated(subset=["ticket_id"], keep=False)])


Shape (rows, columns): (84, 6)

Column types:
ticket_id            object
opened_on            object
channel              object
priority             object
handling_minutes     object
satisfaction        float64
dtype: object

Missing values per column:
ticket_id           1
opened_on           0
channel             1
priority            1
handling_minutes    1
satisfaction        1
dtype: int64

Exact duplicate rows: 3
Repeated ticket IDs (rows involved): 8


,ticket_id,opened_on,channel,priority,handling_minutes,satisfaction
0,T001,2026-10-01,email,medium,42.2,2.0
4,T005,2026-10-02,phone,high,99.5,1.0
9,T010,2026-10-03,email,medium,27.3,1.0
39,T040,2026-10-10,email,medium,52.8,1.0
80,T040,2026-10-10,email,medium,999,1.0
81,T001,2026-10-01,email,medium,42.2,2.0
82,T005,2026-10-02,phone,high,99.5,1.0
83,T010,2026-10-03,email,medium,27.3,1.0


Important observation
In the repeated IDs table, we can see:
- T001, T005 and T010: Each appears twice with identical information. These are exact duplicates.
- T040: Appears twice but has different handling durations: 52.8 and 999.
Why is T040 important?

In the first step, I analyze the dataset. It contains 84 rows and 6 columns. I check the data types, missing values, and duplicates. I find 3 exact duplicates and 8 rows with repeated ticket IDs. I also notice that ticket T040 has conflicting durations. This analysis helps me identify the data quality problems before cleaning.

## 2. Rejection policy

1. Remove **only extra exact copies**.
2. Reject **both/all conflicting occurrences** of an ID (after removing exact copies).
3. Reject missing/malformed IDs and missing, invalid or out-of-October dates.
4. Keep the **original values** of rejected rows and **every applicable reason**.

In [14]:

def reject_bad_rows(data):

    # 1. Make a copy to protect the original dataset
    # Reset the index so rows are numbered from 0
    rows = data.copy(deep=True).reset_index(drop=True)

    # 2. Count exact duplicate rows
    exact_duplicates = int(rows.duplicated().sum())

    # Remove exact duplicates
    rows = rows.drop_duplicates().reset_index(drop=True)

    # 3. Convert ticket IDs and dates into strings
    ids = rows["ticket_id"].astype("string")
    dates_text = rows["opened_on"].astype("string")

    # Convert dates into datetime format
    # Invalid dates become NaT (missing date)
    dates = pd.to_datetime(
        rows["opened_on"],
        errors="coerce",
        format="%Y-%m-%d"
    )

    # 4. Check if ticket ID is missing or empty
    missing_id = ids.isna() | ids.str.strip().eq("").fillna(False)

    # Check if ID follows the format T + 3 digits (example: T001)
    valid_id = ids.str.fullmatch(r"T[0-9]{3}", na=False)

    # Check if the date is missing or empty
    missing_date = dates_text.isna() | dates_text.str.strip().eq("").fillna(False)

    # Check if the date is in October 2026
    in_october = dates.between("2026-10-01", "2026-10-31")

    # 5. Identify all problems in each row
    # True means the row has that problem
    problems = pd.DataFrame({
        "missing_id": missing_id,
        "malformed_id": ~missing_id & ~valid_id,
        "conflicting_id": rows.duplicated(subset=["ticket_id"], keep=False),
        "missing_date": missing_date,
        "invalid_date": ~missing_date & dates.isna(),
        "outside_october": dates.notna() & ~in_october,
    }).fillna(False)

    # 6. Combine all rejection reasons into one text
    # Example: "missing_id; invalid_date"
    reason = problems.apply(
        lambda r: "; ".join(r.index[r]), axis=1
    )

    # Mark rows that have at least one rejection reason
    is_rejected = reason.ne("")

    # 7. Save rejected rows and their rejection reasons
    rejected = rows.loc[is_rejected].copy()
    rejected["rejection_reason"] = reason.loc[is_rejected]

    # 8. Keep only valid rows
    kept = rows.loc[~is_rejected].copy()

    # Store valid dates in datetime format
    kept["opened_on"] = dates.loc[~is_rejected]

    # Reset the index for both datasets
    kept = kept.reset_index(drop=True)
    rejected = rejected.reset_index(drop=True)

    # 9. Return valid rows, rejected rows, and duplicate count
    return kept, rejected, exact_duplicates


# Run the function using the original dataset
kept_2, rejected_2, duplicate_count = reject_bad_rows(raw)

# Show the number of removed duplicates
print("Extra exact copies removed:", duplicate_count)

# Show how many rows were rejected and kept
print("Rejected:", len(rejected_2), " | Kept:", len(kept_2))

# Display rejected tickets with their rejection reasons
display(rejected_2)


Extra exact copies removed: 3
Rejected: 4  | Kept: 77


,ticket_id,opened_on,channel,priority,handling_minutes,satisfaction,rejection_reason
0,T034,2026-10-32,email,medium,7.0,5.0,invalid_date
1,NaN,2026-10-09,phone,high,45.4,1.0,missing_id
2,T040,2026-10-10,email,medium,52.8,1.0,conflicting_id
3,T040,2026-10-10,email,medium,999,1.0,conflicting_id


My function returns three results: the valid tickets, the rejected tickets with their reasons, and the number of exact duplicates removed.

In step two, I created a function called reject_bad_rows. First, I made a copy of the dataset to protect the original data. Then, I removed three exact duplicates. After that, I checked ticket IDs and dates. I found four invalid records: one with an invalid date, one with a missing ID, and two with conflicting IDs. I saved the rejected rows with their reasons and kept 77 valid tickets for the next step.

## 3. Category normalization

Remove extra spaces and use lowercase. Anything not among the allowed values becomes `unknown`, with a Boolean flag.

In [15]:

# Function to clean and standardize categories
def normalize_categories(data):

    # 1. Make a copy to protect the original data
    result = data.copy(deep=True)

    # 2. Define the allowed values for each column
    allowed = {
        "channel": ["email", "phone", "web"],
        "priority": ["low", "medium", "high"]
    }

    # 3. Go through each column and its allowed values
    for column, choices in allowed.items():

        # Convert to text, remove spaces, and use lowercase
        value = result[column].astype("string").str.strip().str.lower()

        # Replace missing or invalid values with "unknown"
        result[column] = value.where(value.isin(choices), "unknown")

        # Create a flag: True if the category is unknown
        result[column + "_unknown"] = result[column].eq("unknown")

    # 4. Return the cleaned dataset
    return result


# Apply the function to the valid tickets from Step 2
kept_3 = normalize_categories(kept_2)

# Count tickets with unknown channels
print("Unknown channel:", kept_3["channel_unknown"].sum())

# Count tickets with unknown priorities
print("Unknown priority:", kept_3["priority_unknown"].sum())

# Display the first 25 tickets with their categories and flags
display(kept_3[["ticket_id", "channel", "channel_unknown",
                "priority", "priority_unknown"]].head(25))


Unknown channel: 2
Unknown priority: 2


,ticket_id,channel,channel_unknown,priority,priority_unknown
0,T001,email,False,medium,False
1,T002,phone,False,high,False
2,T003,web,False,low,False
3,T004,email,False,medium,False
4,T005,phone,False,high,False
5,T006,web,False,low,False
6,T007,email,False,medium,False
7,T008,phone,False,high,False
8,T009,web,False,low,False
9,T010,email,False,medium,False


In step three, I normalize the channel and priority columns. First, I remove extra spaces and convert text to lowercase. Then, I check if each value belongs to the allowed categories. If a value is missing or invalid, I replace it with "unknown" and create a Boolean flag. My results show two unknown channels and two unknown priorities. I keep these tickets because the unknown categories do not require rejecting the entire record.

We don't delete tickets just because their channel or priority is unknown. We replace those categories with "unknown" and preserve that information using flags.

## 4. Numeric treatment

**Handling minutes:** 0 and decimals are valid. Replace missing/invalid/negative/infinite values with the **median of valid retained durations**.

**Satisfaction:** only whole numbers 1–5 are valid. Invalid or missing ratings remain empty: **inventing customer opinions would be misleading**. Extra flags distinguish values missing in the source from values present but invalid.

In [18]:

# Function to clean durations and satisfaction ratings
def treat_numbers(data):

    # 1. Make a copy to protect the original dataset
    result = data.copy(deep=True)

    # PART 1: CLEAN HANDLING DURATION

    # 2. Get the original handling time values
    duration_source = result["handling_minutes"]

    # Convert durations to numbers
    # Invalid text becomes NaN
    duration = pd.to_numeric(
        duration_source, errors="coerce"
    ).astype(float)

    # Check if duration is a real number and is >= 0
    valid_duration = np.isfinite(duration) & duration.ge(0)

    # 3. Identify durations that were originally missing or empty
    result["handling_missing_source"] = duration_source.isna() | (
        duration_source.astype("string").str.strip().eq("").fillna(False)
    )

    # Identify invalid durations (such as -3 or "unknown")
    result["handling_invalid_source"] = (
        ~result["handling_missing_source"] & ~valid_duration
    )

    # Flag all durations that need to be replaced
    result["handling_imputed"] = ~valid_duration

    # 4. Stop if there are no valid durations
    if not valid_duration.any():
        raise ValueError(
            "No valid retained handling_minutes: median imputation is impossible."
        )

    # 5. Calculate the median using only valid durations
    median = float(duration.loc[valid_duration].median())

    # Replace invalid or missing durations with the median
    result["handling_minutes"] = duration.where(valid_duration, median)

    # PART 2: CLEAN SATISFACTION RATINGS

    # 6. Get the original satisfaction values
    rating_source = result["satisfaction"]

    # Convert satisfaction values to numbers
    # Invalid text becomes NaN
    rating = pd.to_numeric(
        rating_source, errors="coerce"
    ).astype(float)

    # 7. Valid ratings must be whole numbers between 1 and 5
    valid_rating = (
        np.isfinite(rating)
        & rating.between(1, 5)
        & rating.mod(1).eq(0)
    )

    # 8. Identify originally missing or empty ratings
    result["satisfaction_missing_source"] = rating_source.isna() | (
        rating_source.astype("string").str.strip().eq("").fillna(False)
    )

    # Identify invalid ratings (such as 7 or 2.5)
    result["satisfaction_invalid_source"] = (
        ~result["satisfaction_missing_source"] & ~valid_rating
    )

    # Flag ratings that are missing or invalid
    result["satisfaction_unavailable"] = ~valid_rating

    # 9. Keep valid ratings and replace invalid ones with NaN
    # We NEVER invent or impute customer satisfaction ratings
    result["satisfaction"] = rating.where(valid_rating, np.nan)

    # 10. Return the cleaned dataset and the median
    return result, median


# Apply numeric treatment to the data from Step 3
kept_4, median_4 = treat_numbers(kept_3)

# Show the median used to fill invalid durations
print("Duration fill median:", median_4)

# Count durations replaced by the median
print("Imputed durations:", kept_4["handling_imputed"].sum())

# Count missing or invalid satisfaction ratings
print("Unavailable satisfaction ratings:",
      kept_4["satisfaction_unavailable"].sum())

# Display tickets with corrected durations or unavailable ratings
display(
    kept_4.loc[
        kept_4["handling_imputed"] | kept_4["satisfaction_unavailable"],
        ["ticket_id", "handling_minutes", "handling_imputed",
         "satisfaction", "satisfaction_unavailable"]
    ]
)


Duration fill median: 59.0
Imputed durations: 3
Unavailable satisfaction ratings: 3


,ticket_id,handling_minutes,handling_imputed,satisfaction,satisfaction_unavailable
2,T003,59.0,True,4.0,False
6,T007,59.0,True,3.0,False
10,T011,59.0,True,2.0,False
26,T027,81.4,False,NaN,True
28,T029,22.0,False,NaN,True
29,T030,33.4,False,NaN,True


In step four, I clean the numerical columns: handling minutes and satisfaction. For handling minutes, I accept only finite and non-negative numbers. I replace missing or invalid durations with the median, which is 59 minutes. Three durations are corrected. For satisfaction, I accept only whole numbers between 1 and 5. I found three unavailable ratings, which I keep as NaN because I should not invent customer feedback.

## 5. Features

Standardize the repaired duration with **population** standard deviation (`ddof=0`). If all durations are identical, divide by **1**, so all z-scores become 0. Create exactly four one-hot channel indicators. Do **not** use satisfaction as a modelling feature.

In [19]:

# Function to create new features for Machine Learning
def make_features(data):

    # 1. Make a copy to protect the original dataset
    result = data.copy(deep=True)

    # 2. Calculate the average handling duration
    mean = float(result["handling_minutes"].mean())

    # Calculate the population standard deviation
    std = float(result["handling_minutes"].std(ddof=0))

    # 3. Avoid division by zero if all durations are equal
    denominator = std if std > 0 else 1.0

    # 4. Standardize handling duration using the Z-score formula
    # Z-score = (duration - mean) / standard deviation
    result["handling_zscore"] = (
        result["handling_minutes"] - mean
    ) / denominator

    # 5. Create a separate table with ticket ID and Z-score
    features = result[["ticket_id", "handling_zscore"]].copy()

    # 6. Convert each channel category into a numerical column
    # 1 means the ticket belongs to that channel
    # 0 means it does not
    for name in ["email", "phone", "web", "unknown"]:
        features["channel_" + name] = (
            result["channel"].eq(name).astype(int)
        )

    # 7. Verify that ticket IDs have the same order
    assert features["ticket_id"].equals(result["ticket_id"])

    # 8. Verify that every ticket belongs to exactly one channel
    assert (
        features[
            ["channel_email", "channel_phone",
             "channel_web", "channel_unknown"]
        ].sum(axis=1) == 1
    ).all()

    # 9. Return the updated data, features, mean, and std
    return result, features, mean, std


# Apply feature engineering to the cleaned dataset
kept_5, features_5, mean_5, std_5 = make_features(kept_4)

# Display the calculated mean and standard deviation
print(
    "Duration mean:", round(mean_5, 3),
    " | Population std:", round(std_5, 3)
)

# Display the first 5 rows of the feature table
display(features_5.head())


Duration mean: 59.59  | Population std: 30.595


,ticket_id,handling_zscore,channel_email,channel_phone,channel_web,channel_unknown
0,T001,-0.568376,1,0,0,0
1,T002,0.546176,0,1,0,0
2,T003,-0.019271,0,0,1,0
3,T004,1.909131,1,0,0,0
4,T005,1.304463,0,1,0,0


In step five, I create features for Machine Learning. First, I calculate the mean and standard deviation of handling durations. Then, I use them to calculate a Z-score for each ticket. I also convert the channel categories into four numerical columns using one-hot encoding. Finally, I use assertions to verify that ticket IDs are aligned and each ticket belongs to exactly one channel.

## 6. Reusable function: `clean_tickets(raw)`

This function **reuses the four small steps above**, returns all required objects, and checks that the data satisfies the lecturer's acceptance rules.

In [20]:

# Main function: combine all cleaning steps
def clean_tickets(data):

    # 1. Remove exact duplicates and reject invalid records
    # The original dataset is not modified
    kept, rejected, n_duplicates = reject_bad_rows(data)

    # 2. Normalize channel and priority categories
    kept = normalize_categories(kept)

    # 3. Clean durations and satisfaction ratings
    # Get the median used to replace invalid durations
    kept, fill_median = treat_numbers(kept)

    # 4. Create Z-scores and channel indicators
    # Also calculate the mean and standard deviation
    clean, features, mean, std = make_features(kept)

    # 5. Create an audit dictionary to summarize the cleaning
    audit = {

        # Number of rows in the original dataset
        "raw_rows": int(len(data)),

        # Number of exact duplicate copies removed
        "exact_duplicates_removed": int(n_duplicates),

        # Number of rejected rows
        "rejected_rows": int(len(rejected)),

        # Number of valid tickets remaining
        "retained_rows": int(len(clean)),

        # Count tickets with unknown channels
        "channel_unknown": int(clean["channel_unknown"].sum()),

        # Count tickets with unknown priorities
        "priority_unknown": int(clean["priority_unknown"].sum()),

        # Count durations replaced by the median
        "handling_imputed": int(clean["handling_imputed"].sum()),

        # Count originally missing durations
        "handling_missing_source": int(
            clean["handling_missing_source"].sum()
        ),

        # Count originally invalid durations
        "handling_invalid_source": int(
            clean["handling_invalid_source"].sum()
        ),

        # Count missing or invalid satisfaction ratings
        "satisfaction_unavailable": int(
            clean["satisfaction_unavailable"].sum()
        ),

        # Count originally missing satisfaction ratings
        "satisfaction_missing_source": int(
            clean["satisfaction_missing_source"].sum()
        ),

        # Count originally invalid satisfaction ratings
        "satisfaction_invalid_source": int(
            clean["satisfaction_invalid_source"].sum()
        ),

        # Median used to fill invalid durations
        "handling_fill_median": fill_median,

        # Average duration after cleaning
        "handling_mean": mean,

        # Population standard deviation after cleaning
        "handling_std_population": std,
    }

    # PART 2: ACCEPTANCE CHECKS

    # 6. Check that all original rows are accounted for
    assert audit["raw_rows"] == (
        audit["exact_duplicates_removed"] +
        audit["rejected_rows"] +
        audit["retained_rows"]
    )

    # 7. Check that all IDs follow the format T + 3 digits
    assert clean["ticket_id"].str.fullmatch(r"T[0-9]{3}").all()

    # Check that each ticket ID is unique
    assert clean["ticket_id"].is_unique

    # 8. Check that all dates are in October 2026
    assert clean["opened_on"].between(
        "2026-10-01", "2026-10-31"
    ).all()

    # 9. Check that all handling durations are finite numbers
    assert np.isfinite(clean["handling_minutes"]).all()

    # Check that no handling duration is negative
    assert clean["handling_minutes"].ge(0).all()

    # 10. Check that channels contain only allowed values
    assert clean["channel"].isin(
        ["email", "phone", "web", "unknown"]
    ).all()

    # Check that priorities contain only allowed values
    assert clean["priority"].isin(
        ["low", "medium", "high", "unknown"]
    ).all()

    # 11. Select only available satisfaction ratings
    observed = clean["satisfaction"].dropna()

    # Check that available ratings are whole numbers from 1 to 5
    assert (
        observed.between(1, 5) &
        observed.mod(1).eq(0)
    ).all()

    # 12. Check that feature IDs match the cleaned ticket IDs
    assert features["ticket_id"].equals(clean["ticket_id"])

    # Define the four channel indicator columns
    indicator_cols = [
        "channel_email",
        "channel_phone",
        "channel_web",
        "channel_unknown"
    ]

    # Check that each ticket has exactly one active channel
    assert features[indicator_cols].sum(axis=1).eq(1).all()

    # 13. Check that the average Z-score is approximately zero
    assert np.isclose(
        features["handling_zscore"].mean(),
        0,
        atol=1e-10
    )

    # If standard deviation is positive,
    # check that the Z-score standard deviation is approximately 1
    if std > 0:
        assert np.isclose(
            features["handling_zscore"].std(ddof=0),
            1,
            atol=1e-10
        )

    # If all durations are equal, all Z-scores must be zero
    else:
        assert (features["handling_zscore"] == 0).all()

    # 14. Return the four final results
    return clean, rejected, features, audit


# PART 3: RUN THE COMPLETE PIPELINE

# Run the function on the original dataset
clean, rejected, features, audit = clean_tickets(raw)

# Verify that the original dataset has not changed
# raw_before is a copy saved before running the pipeline
assert raw.equals(raw_before), (
    "The original DataFrame must stay unchanged."
)

# Print success message if all checks passed
print("All acceptance checks passed. Original input unchanged.")

# Display the audit summary in a readable JSON format
print(json.dumps(audit, indent=2))

# Display the first five cleaned tickets
display(clean.head())


All acceptance checks passed. Original input unchanged.
{
  "raw_rows": 84,
  "exact_duplicates_removed": 3,
  "rejected_rows": 4,
  "retained_rows": 77,
  "channel_unknown": 2,
  "priority_unknown": 2,
  "handling_imputed": 3,
  "handling_missing_source": 1,
  "handling_invalid_source": 2,
  "satisfaction_unavailable": 3,
  "satisfaction_missing_source": 1,
  "satisfaction_invalid_source": 2,
  "handling_fill_median": 59.0,
  "handling_mean": 59.589610389610385,
  "handling_std_population": 30.595274163617248
}


,ticket_id,opened_on,channel,priority,handling_minutes,satisfaction,channel_unknown,priority_unknown,handling_missing_source,handling_invalid_source,handling_imputed,satisfaction_missing_source,satisfaction_invalid_source,satisfaction_unavailable,handling_zscore
0,T001,2026-10-01,email,medium,42.2,2.0,False,False,False,False,False,False,False,False,-0.568376
1,T002,2026-10-01,phone,high,76.3,3.0,False,False,False,False,False,False,False,False,0.546176
2,T003,2026-10-01,web,low,59.0,4.0,False,False,True,False,True,False,False,False,-0.019271
3,T004,2026-10-01,email,medium,118.0,5.0,False,False,False,False,False,False,False,False,1.909131
4,T005,2026-10-02,phone,high,99.5,1.0,False,False,False,False,False,False,False,False,1.304463


In step six, I combine all the previous functions into one reusable function called clean_tickets.

First, it removes duplicates and rejects invalid records. Then it normalizes categories, treats numerical values, and creates features.

I also create an audit dictionary to track all changes made to the data.

Finally, I use assertions to verify data quality and make sure the original dataset remains unchanged.

The pipeline starts with 84 rows, removes 3 exact duplicates, rejects 4 invalid records, and keeps 77 valid tickets.

All acceptance checks passed.

## 7. Edge cases

Small tests prove that the function works on **other datasets**, not only this file. Each `assert` must pass.

In [22]:

# STEP 7: TEST EDGE CASES

# TEST A: Constant duration and missing value

# Create a small dataset with only 2 tickets
two = pd.DataFrame({
    "ticket_id": ["T901", "T902"],
    "opened_on": ["2026-10-01", "2026-10-02"],
    "channel": ["email", "web"],
    "priority": ["low", "high"],
    "handling_minutes": [10, None],
    "satisfaction": [5, 4],
})

# Run our cleaning function
c, r, f, a = clean_tickets(two)

# Check that the missing duration is replaced with 10
assert c["handling_minutes"].tolist() == [10, 10]

# Check that only the second duration was imputed
assert c["handling_imputed"].tolist() == [False, True]

# Both durations are equal, so both Z-scores must be 0
assert f["handling_zscore"].tolist() == [0, 0]

# No tickets should be rejected
assert len(r) == 0


# TEST B: Exact duplicate

# Add an exact copy of the first ticket
dup_test = pd.concat([two, two.iloc[[0]]], ignore_index=True)

# Run the cleaning function
_, r, _, a = clean_tickets(dup_test)

# Check that only one duplicate copy was removed
# No rows should be rejected
assert a["exact_duplicates_removed"] == 1 and len(r) == 0


# TEST C: Conflicting ticket IDs

# Create 2 tickets with the same ID but different information
# One also has a date outside October 2026
conflicts = pd.DataFrame({
    "ticket_id": ["T910", "T910", "T911"],
    "opened_on": ["2026-10-01", "2026-11-01", "2026-10-03"],
    "channel": ["email", "phone", "web"],
    "priority": ["low", "high", "low"],
    "handling_minutes": [20, 21, 10],
    "satisfaction": [3, 4, 5],
})

# Run the cleaning function
c, r, _, a = clean_tickets(conflicts)

# Check that 1 ticket is kept and 2 are rejected
assert len(c) == 1 and len(r) == 2

# Both rejected tickets must have a conflicting ID reason
assert r["rejection_reason"].str.contains("conflicting_id").all()

# One rejected ticket must also have an outside-October reason
assert r["rejection_reason"].str.contains("outside_october").sum() == 1

# Check that all original rows are accounted for
assert a["raw_rows"] == (
    a["exact_duplicates_removed"] +
    a["rejected_rows"] +
    a["retained_rows"]
)


# TEST D: Invalid dates

# Add three tickets with problematic dates
date_test = pd.concat([two, pd.DataFrame({
    "ticket_id": ["T903", "T904", "T905"],
    "opened_on": [None, "not-a-date", "2026-09-30"],
    "channel": ["web"] * 3,
    "priority": ["low"] * 3,
    "handling_minutes": [10] * 3,
    "satisfaction": [5] * 3,
})], ignore_index=True)

# Run the cleaning function
c, r, _, a = clean_tickets(date_test)

# Check that all 3 invalid-date tickets are rejected
assert len(r) == 3

# Check that all 3 rejection reasons are recorded
assert set(r["rejection_reason"]) == {
    "missing_date",
    "invalid_date",
    "outside_october"
}


# TEST E: Special numeric values

# Copy the 2-ticket dataset
special = two.copy()

# Test zero and decimal durations
special["handling_minutes"] = [0, 2.5]

# Test a decimal satisfaction rating
special["satisfaction"] = [2.5, 4]

# Run the cleaning function
c, r, _, _ = clean_tickets(special)

# Both durations are valid, so no imputation is needed
assert c["handling_imputed"].sum() == 0

# Rating 2.5 is invalid, but rating 4 is valid
assert c["satisfaction_unavailable"].tolist() == [True, False]

# Check that the invalid rating becomes NaN
assert pd.isna(c.loc[0, "satisfaction"])


# TEST F: Missing and malformed ticket IDs

# Add 2 tickets: one with a missing ID and one with a wrong ID
invalid_ids = pd.concat([two, pd.DataFrame({
    "ticket_id": [None, "ticket-123"],
    "opened_on": ["2026-10-02", "2026-10-03"],
    "channel": ["phone", "email"],
    "priority": ["high", "low"],
    "handling_minutes": [15, 16],
    "satisfaction": [3, 4],
})], ignore_index=True)

# Run the cleaning function
c, r, _, _ = clean_tickets(invalid_ids)

# Check that 2 valid tickets remain and 2 are rejected
assert len(c) == 2 and len(r) == 2

# Verify that both rejection reasons are recorded
assert set(r["rejection_reason"]) == {
    "missing_id",
    "malformed_id"
}


# TEST G: No valid handling duration

# Copy the 2-ticket dataset
bad = two.copy()

# Make both durations invalid
bad["handling_minutes"] = [None, -2]

# Try to clean the dataset
try:
    clean_tickets(bad)

    # Fail the test if the expected error was not raised
    raise AssertionError("Expected ValueError for no valid duration")

# Catch the expected ValueError
except ValueError as error:

    # Check that the error contains the correct message
    assert "No valid retained handling_minutes" in str(error)


# Print a success message if all 7 tests pass
print("PASS: constant duration, exact duplicates, conflicting IDs, multiple reasons,")
print("      dates, invalid IDs, zero/fractional values, and no-valid-duration error.")


PASS: constant duration, exact duplicates, conflicting IDs, multiple reasons,
      dates, invalid IDs, zero/fractional values, and no-valid-duration error.


In step seven, I test my cleaning function using seven different edge cases.

I test missing durations, exact duplicates, conflicting ticket IDs, invalid dates, special numeric values, and malformed IDs.

I also test what happens when no valid handling duration exists.

I use assertions to check that the function produces the expected results.

All seven tests passed, which shows that the pipeline handles these situations correctly.

## 8. Export and report

Save **all required files** and read the three CSV files back. `REPORT.md` is in English and includes limitations, run instructions, and an AI-assistance disclosure.

In [9]:
outputs = Path("outputs")
outputs.mkdir(exist_ok=True)

clean.to_csv(outputs / "clean_tickets.csv", index=False, date_format="%Y-%m-%d")
rejected.to_csv(outputs / "rejected_tickets.csv", index=False)
features.to_csv(outputs / "ticket_features.csv", index=False)
(outputs / "audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")

# Read all exported files back and compare the important properties.
clean_read = pd.read_csv(outputs / "clean_tickets.csv")
rejected_read = pd.read_csv(outputs / "rejected_tickets.csv")
features_read = pd.read_csv(outputs / "ticket_features.csv")
audit_read = json.loads((outputs / "audit.json").read_text(encoding="utf-8"))

assert len(raw) == audit_read["exact_duplicates_removed"] + len(rejected_read) + len(clean_read)
assert len(clean_read) == len(features_read)
assert clean_read["ticket_id"].tolist() == features_read["ticket_id"].tolist()
assert clean_read["ticket_id"].is_unique
assert features_read[["channel_email", "channel_phone", "channel_web",
                      "channel_unknown"]].sum(axis=1).eq(1).all()
assert clean_read["handling_minutes"].notna().all()
assert clean_read["satisfaction"].isna().sum() == audit_read["satisfaction_unavailable"]
assert raw.equals(raw_before)

report = f"""# REPORT — Service Ticket Data Pipeline

**Student:** Mohamed Jlidi (a22601339)
**Course:** Applied Programming for AI — Mini-project 1

## Diagnosis
The unchanged synthetic input contains **{len(raw)} rows and {len(raw.columns)} columns**. Each row is one service-ticket observation. The initial examination inspected the column types, missing values, exact copies, and repeated ticket identifiers. A ticket ID should identify one unique ticket, but repeated IDs can have conflicting data.

## Rejection policy
I removed **{audit['exact_duplicates_removed']} extra exact copies**. I then rejected **{audit['rejected_rows']} rows** whose ID was missing or malformed, whose ID conflicted with another remaining record, or whose date was missing, invalid, or outside 1–31 October 2026. When two rows shared a conflicting ID, I rejected both. Every rejected source row was preserved with all applicable reasons in `rejected_tickets.csv`.

## Cleaning and features
I kept **{audit['retained_rows']} trustworthy ticket rows**. Category values were stripped and lowercased, while unrecognized or missing categories became `unknown` with flags ({audit['channel_unknown']} channel, {audit['priority_unknown']} priority). Duration values may be zero or fractional, but cannot be negative, infinite, or unparseable. I replaced **{audit['handling_imputed']} invalid/missing durations** with the median **{audit['handling_fill_median']:.2f} minutes** of valid retained durations. The extra source flags distinguish missing values from present-but-invalid values. The **{audit['satisfaction_unavailable']} unavailable satisfaction ratings** stayed missing because filling them would invent customer opinions. I created a duration z-score with population standard deviation and four one-hot channel indicators.

## Checks, interpretation, limitations
All assertions passed: **{audit['raw_rows']} = {audit['exact_duplicates_removed']} + {audit['rejected_rows']} + {audit['retained_rows']}**. IDs are unique, dates and durations are valid, each feature row matches the clean table, and exactly one channel indicator is active. Tests also covered constant durations, duplicates, conflicting IDs, date errors, fractional inputs, and the missing-median error.

**Limitation 1:** This small synthetic dataset does not represent real support-service patterns. **Limitation 2:** Replacing invalid durations with a median reduces variation and can hide unusual cases. For a future predictive model, normalization statistics must be learned only from the training set to avoid data leakage.

## How to run and assistance used
Open `project_1_student.ipynb` in Google Colab, upload the unchanged `service_tickets.csv` if prompted, and choose **Runtime → Run all**. The notebook creates `outputs/` and verifies each saved CSV. I used ChatGPT to help structure, explain, and test the Python solution; I reviewed the rules and must be able to explain the code during the individual oral defence.
"""
Path("REPORT.md").write_text(report, encoding="utf-8")

readme = """# Applied Programming for AI — Mini-project 1

**Student:** Mohamed Jlidi
**Student number:** a22601339

## Files
- `project_1_student.ipynb` — complete 8-step notebook.
- `service_tickets.csv` — unchanged synthetic input.
- `outputs/clean_tickets.csv` — cleaned ticket data and flags.
- `outputs/rejected_tickets.csv` — rejected source rows and reasons.
- `outputs/ticket_features.csv` — numeric channel indicators and duration z-scores.
- `outputs/audit.json` — counts, median, mean, and population standard deviation.
- `REPORT.md` — methods, tests, limitations, and assistance disclosure.

## Run from a fresh clone / in Colab
Requires Python 3, NumPy, Pandas, and Jupyter (all present in Google Colab).

1. Open `project_1_student.ipynb` using Google Colab (or Jupyter).
2. Keep `service_tickets.csv` in the same folder; if in Colab, upload it when prompted.
3. Choose **Runtime → Run all**.
4. The last cells regenerate the required files and verify the exports.

No changes to the original CSV, model training, or scikit-learn are required.

## Submission
Commit the notebook, original CSV, `REPORT.md`, `README.md`, and the entire `outputs/` directory to your own GitHub repository. Invite GitHub user `tgvp` as a collaborator. The deadline is announced separately by the lecturer.
"""
Path("README.md").write_text(readme, encoding="utf-8")

print("Export checks passed. Generated files:")
for p in [Path("README.md"), Path("REPORT.md"), *sorted(outputs.iterdir())]:
    print(" -", p)
print("REPORT.md word count:", len(report.split()))

Export checks passed. Generated files:
 - README.md
 - REPORT.md
 - outputs/audit.json
 - outputs/clean_tickets.csv
 - outputs/rejected_tickets.csv
 - outputs/ticket_features.csv
REPORT.md word count: 382


### Optional: download a submission ZIP directly from Google Colab

The next cell packages the notebook, input data, report and exported outputs. Colab users can download it, then upload its contents to a personal GitHub repository. If the notebook is being run elsewhere, the ZIP will still be created.

In [10]:
from zipfile import ZipFile

zip_name = "service_ticket_submission.zip"
files_to_zip = [Path("service_tickets.csv"), Path("REPORT.md"), Path("README.md")]
files_to_zip += sorted(Path("outputs").glob("*"))

# Colab normally stores the active notebook outside the runtime filesystem.
# Include a notebook file when it exists locally; otherwise download the notebook
# from Colab via File > Download > Download .ipynb and add it to your repository.
local_notebook = Path("project_1_student.ipynb")
if local_notebook.is_file():
    files_to_zip.insert(0, local_notebook)

with ZipFile(zip_name, "w") as z:
    for p in files_to_zip:
        z.write(p, arcname=p.as_posix())
print("Created:", zip_name)
if not local_notebook.is_file():
    print("Remember: File > Download > Download .ipynb in Colab, then add the notebook to GitHub.")

try:
    from google.colab import files
    files.download(zip_name)
except ImportError:
    pass

Created: service_ticket_submission.zip
Remember: File > Download > Download .ipynb in Colab, then add the notebook to GitHub.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>